# 26/ Stock Price Prediction

**Dataset (Kaggle):** [https://www.kaggle.com/datasets/camnugent/sandp500](https://www.kaggle.com/datasets/camnugent/sandp500)

> 5 years of daily OHLCV data for all S&P 500 companies in a single all_stocks_5yr.csv (columns: date, open, high, low, close, volume, Name).

**Goal:** Forecast a stock's next closing price using an LSTM sequence model on historical prices.

**How to get the data:**
1. Download from the link above (or run `kaggle datasets download -d <dataset-slug> -p data/ --unzip`, requires a `kaggle.json` API token in `~/.kaggle/`).
2. Place the extracted file(s) in a local `data/` folder next to this notebook.
3. Update `DATA_PATH` in the cell below if your filename differs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, models

sns.set_style("whitegrid")


In [ ]:
DATA_PATH = "data/all_stocks_5yr.csv"
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
print(df['Name'].nunique(), "tickers available")

TICKER = "AAPL"  # change to any ticker present in df['Name']
stock = df[df['Name'] == TICKER].sort_values('date').reset_index(drop=True)
stock.head()


## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(stock['date'], stock['close'])
plt.title(f"{TICKER} Closing Price")
plt.xlabel("Date"); plt.ylabel("Close Price")
plt.show()


## Preprocessing: Scale & Create Sequences

In [ ]:
close_prices = stock[['close']].values
scaler = MinMaxScaler()
scaled = scaler.fit_transform(close_prices)

WINDOW = 60

def make_sequences(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = make_sequences(scaled, WINDOW)
X = X.reshape((X.shape[0], X.shape[1], 1))

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(X_train.shape, X_test.shape)


## Build LSTM Model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(WINDOW, 1)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(64),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.summary()


## Train

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)


## Evaluate

In [ ]:
pred_scaled = model.predict(X_test)
pred = scaler.inverse_transform(pred_scaled)
actual = scaler.inverse_transform(y_test.reshape(-1,1))

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
print(f"MAE: {mae:.2f}   RMSE: {rmse:.2f}")

plt.figure(figsize=(12,4))
plt.plot(actual, label='Actual')
plt.plot(pred, label='Predicted')
plt.title(f"{TICKER}: Actual vs Predicted Close Price")
plt.legend()
plt.show()


## ⚠️ Disclaimer
This notebook is an educational demonstration of time-series modeling. Stock prices are highly noisy and influenced by countless external factors; this model should **not** be used for real trading decisions.

## Next Steps
- Add technical indicators (moving averages, RSI, MACD) as extra features.
- Try a Prophet or ARIMA baseline for comparison.
- Backtest a simple trading strategy and compare to buy-and-hold.